## 1) Install required libraries

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn

## 2) Import libraries

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## 3) Load the dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
df.head()

## 4) Quick inspection

In [ ]:
print('Shape:', df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)

## 5) Expected columns

In [ ]:
print('Columns:', df.columns.tolist())

## 6) Basic cleaning


In [ ]:
df = df.copy()

df.columns = [c.strip().lower() for c in df.columns]

df = df.drop_duplicates()

for col in df.columns:
    if col != 'country':
        df[col] = pd.to_numeric(df[col], errors='coerce')

numeric_cols = [c for c in df.columns if c != 'country']

df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

print(df.isna().sum())

## 7) Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)
plt.title('Correlation Heatmap')
plt.show()

## 8) Feature scaling


In [ ]:
features = df.drop(columns=['country'], errors='ignore')

scaler = StandardScaler()

X_scaled = scaler.fit_transform(features)

print(X_scaled.shape)

## 9) K-Means: Elbow method

In [ ]:
inertias = []
k_values = range(2, 11)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8,4))
plt.plot(list(k_values), inertias, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.show()

## 10) Train K-Means

In [ ]:
best_k = 3

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

print('Silhouette Score:',
      silhouette_score(X_scaled, df['kmeans_cluster']))

df[['country', 'kmeans_cluster']].head()

## 11) DBSCAN

In [ ]:
dbscan = DBSCAN(
    eps=1.5,
    min_samples=5
)

df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

print(
    df['dbscan_cluster']
    .value_counts()
    .sort_index()
)

## 12) PCA visualization

In [ ]:
pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(X_scaled)

viz = pd.DataFrame({
    'pca1': X_pca[:,0],
    'pca2': X_pca[:,1],
    'cluster': df['kmeans_cluster']
})

plt.figure(figsize=(8,6))

sns.scatterplot(
    data=viz,
    x='pca1',
    y='pca2',
    hue='cluster',
    palette='tab10'
)

plt.title(
    'K-Means Clusters Visualized with PCA'
)

plt.show()

## 13) Cluster profiling

In [ ]:
profile = df.groupby('kmeans_cluster')[numeric_cols].mean().round(2)
profile

## 14) Random Forest (Classification on clusters)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = df[numeric_cols]
y = df['kmeans_cluster']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))

## 15) XGBoost

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

## 16) Final insights


In [ ]:

print("1. Cluster 1 has high child mortality and low life expectancy, indicating poor health conditions.")
print("2. Cluster 0 has high income and high GDPP, representing economically strong countries.")
print("3. Cluster 1 looks underdeveloped due to low income, low GDP, and high child mortality.")
print("4. Countries in Cluster 1 should be prioritized for aid.")
print("5. K-Means clustering effectively separated developed and underdeveloped countries.")